# CS2 Lecture 3 — optional

안 풀어도 됨. numpy·pandas로 다차원 리스트 다뤄보는 것. 빈칸(`# ADD`)만 채우면 된다.

## 0. numpy 워밍업

`np.array` 는 다차원 리스트처럼 생겼는데 반복문 없이 통째로 계산된다.
blank 4개를 채워보자.

In [ ]:
import numpy as np

A = np.arange(12).reshape(3, 4)
print(A)

col1   = None   # ADD: A 의 1번 열        -> [1, 5, 9]
rowsum = None   # ADD: A 각 행의 합        -> [6, 22, 38]
big    = None   # ADD: A 에서 5 초과 개수  -> 6
block  = None   # ADD: A 의 행 1~2, 열 1~2 -> [[5, 6], [9, 10]]

assert col1.tolist()   == [1, 5, 9]
assert rowsum.tolist() == [6, 22, 38]
assert int(big)        == 6
assert block.tolist()  == [[5, 6], [9, 10]]
print("OK")

## 1. 이미지 흐리기 (box blur)

흑백 이미지 = `H×W` 2차원 리스트, `g[i][j]` 는 밝기 `0`~`255`. Problem 6에서
각 칸의 3×3 이웃에 있는 지뢰 수를 셌는데, 이번엔 이웃의 **평균**을 낸다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_sample_image

rgb = load_sample_image("china.jpg")
g = (rgb @ [0.299, 0.587, 0.114]).round().astype(int).tolist()   # H x W 2차원 리스트
g = [row[250:450] for row in g[100:250]]                         # 150 x 200 로 자르기

def blur(g):
    H, W = len(g), len(g[0])
    out = [[0] * W for _ in range(H)]
    for i in range(H):
        for j in range(W):
            total = count = 0
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    ni, nj = i + di, j + dj
                    if 0 <= ni < H and 0 <= nj < W:
                        pass  # ADD: total, count 갱신
            out[i][j] = total / count
    return out

plt.imshow(blur(g), cmap="gray")
plt.show()

## 2. 흰 배경에서 객체만 잘라내기

배경이 흰색(≈`255`)인 이미지에서, 어두운 픽셀을 다 감싸는 가장 작은
직사각형만 남긴다. 어두운 픽셀들의 행/열 인덱스 min·max 를 찾으면 끝
(Problem 7 `maxZeroRect` 와 같은 얘기).

In [ ]:
import matplotlib.pyplot as plt

# 흰 배경(255) + 어두운 사각형 하나
g = [[255] * 60 for _ in range(40)]
for i in range(10, 25):
    for j in range(15, 45):
        g[i][j] = 80

def crop_object(g, thresh=250):
    H, W = len(g), len(g[0])
    min_i, max_i, min_j, max_j = H, -1, W, -1
    for i in range(H):
        for j in range(W):
            if g[i][j] < thresh:
                pass  # ADD: min_i, max_i, min_j, max_j 갱신
    # 찾은 직사각형만 잘라서 반환
    return [row[min_j:max_j + 1] for row in g[min_i:max_i + 1]]

crop = crop_object(g)
print(len(g), "x", len(g[0]), " -> ", len(crop), "x", len(crop[0]))   # 40 x 60  ->  15 x 30

# numpy 라면 한 줄:  np.array(g)[min_i:max_i+1, min_j:max_j+1]
plt.imshow(crop, cmap="gray", vmin=0, vmax=255)
plt.show()

## 3. 주가 = 시간 × 종목 2차원 리스트

인터넷에서 주가 표를 받아 한 종목의 종가 리스트를 만든다. 20일 이동평균을
구한다 — 시간축으로 이웃 평균이니 1번 blur 의 1차원 버전이다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/plotly/datasets/master/stockdata.csv"
try:
    price = pd.read_csv(URL)["AAPL"].tail(300).tolist()
except Exception:
    price = (100 + np.random.default_rng(0).normal(0, 1, 300).cumsum()).tolist()

def moving_avg(col, w):
    out = [0.0] * len(col)
    for t in range(len(col)):
        lo = max(0, t - w + 1)
        pass  # ADD: col[lo..t] 의 평균을 out[t] 에
    return out

plt.plot(price, lw=.8)
plt.plot(moving_avg(price, 20))
plt.show()